<a href="https://colab.research.google.com/github/JBEstevan/ai-code-mentor/blob/main/ai_code_mentor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU langchain langchain-community langchain-google-genai pypdf chromadb sentence-transformers "numpy<2.0.0"
print("Bibliotecas instaladas.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 103.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.5.4 requires google-ai-generativelanguage==0.6.4, but you have google-ai-generativelanguage 0.12.0 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7,

In [4]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

chave = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = chave

print("1. Lendo o manual de Boas Práticas e Clean Code...")
loader = PyPDFLoader("Boas_Praticas_de_Codigo_e_Clean_Code.pdf")
documentos = loader.load()

print("2. Fatiando o texto para a IA entender...")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
pedacos_texto = text_splitter.split_documents(documentos)

print("3. Criando o banco de dados vetorial (ChromaDB com HuggingFace)...")

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
banco_vetorial = Chroma.from_documents(pedacos_texto, embeddings)
retriever = banco_vetorial.as_retriever()

print("4. Iniciando o Mentor (Gemini 3.6 Flash)...")

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.3)

system_prompt = (
    "Você é um Engenheiro de Software Sênior e Mentor de Backend. "
    "Use os trechos de contexto a seguir extraídos do nosso manual de boas práticas "
    "para responder à pergunta do desenvolvedor júnior. "
    "Seja didático, encorajador e dê exemplos de código se possível. "
    "Se a resposta não estiver no contexto, diga educadamente que o manual não cobre esse tópico.\n\n"
    "Contexto:\n{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
agente = create_retrieval_chain(retriever, question_answer_chain)

print("\nMentor pronto! Vamos começar:")
print("-" * 50)

pergunta = "De acordo com o manual, o que caracteriza a Regra do Escoteiro (Boy Scout Rule) e qual a sua importância?"
print(f"Pergunta do Dev: {pergunta}\n")

resposta = agente.invoke({"input": pergunta})

print("Resposta do Mentor:")
print(resposta["answer"])

1. Lendo o manual de Boas Práticas e Clean Code...
2. Fatiando o texto para a IA entender...
3. Criando o banco de dados vetorial (ChromaDB com HuggingFace)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

4. Iniciando o Mentor (Gemini 3.6 Flash)...

Mentor pronto! Vamos começar:
--------------------------------------------------
Pergunta do Dev: De acordo com o manual, o que caracteriza a Regra do Escoteiro (Boy Scout Rule) e qual a sua importância?

Resposta do Mentor:
Olá! Que excelente pergunta. Essa é uma das diretrizes mais práticas e valiosas para o nosso dia a dia como desenvolvedores backend.

De acordo com o nosso manual, a **Regra do Escoteiro (Boy Scout Rule)** é caracterizada e justificada da seguinte forma:

---

### 1. O que caracteriza a Regra do Escoteiro?
A regra se resume a um princípio simples: **sempre deixar o código um pouco mais limpo do que quando você o encontrou**. 

Significa que, ao abrir um arquivo para corrigir um *bug* ou implementar uma nova funcionalidade, se você notar um trecho confuso, uma variável mal nomeada ou uma pequena sujeira no código, você deve aproveitá-lo para aplicar uma melhoria pontual.

### 2. Qual a sua importância?
A importância dessa